# PySpark Neural Network in Google Colab — Fixed with Hadoop/Spark Environment Setup

This notebook rewrites the original neural-network notebook using the same reliable environment setup style
as the working Hadoop example:

- explicit Java installation
- explicit Spark download
- Spark distribution bundled with Hadoop support
- subprocess instead of fragile notebook shell magics
- environment variables set before Spark startup

It keeps the original learning goal:
- build a neural network in PySpark
- use MultilayerPerceptronClassifier
- train on a small Iris-style dataset
- generate predictions
- evaluate accuracy and F1
- export results to CSV


In [ ]:
# Step 1: Install Java + Spark (with Hadoop package/environment style)
import os
import subprocess
from pathlib import Path

SPARK_VERSION = "3.5.1"
HADOOP_FLAVOR = "hadoop3"
SPARK_DIR = f"/content/spark-{SPARK_VERSION}-bin-{HADOOP_FLAVOR}"
SPARK_TAR = f"/content/spark-{SPARK_VERSION}-bin-{HADOOP_FLAVOR}.tgz"
SPARK_URL = f"https://archive.apache.org/dist/spark/spark-{SPARK_VERSION}/spark-{SPARK_VERSION}-bin-{HADOOP_FLAVOR}.tgz"

def run(cmd, env=None):
    print("RUN:", cmd)
    completed = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {cmd}")
    return completed

run("apt-get -qq update")
run("apt-get install -y openjdk-11-jdk-headless wget tar > /dev/null")
run("pip -q install pyspark pandas matplotlib")

if not Path(SPARK_DIR).exists():
    run(f"wget -q -O {SPARK_TAR} {SPARK_URL}")
    run(f"tar -xzf {SPARK_TAR} -C /content/")
else:
    print("Spark already installed. Skipping download/extract.")

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = SPARK_DIR
os.environ["HADOOP_HOME"] = SPARK_DIR
os.environ["PATH"] = os.environ["PATH"] + f":{SPARK_DIR}/bin:{SPARK_DIR}/sbin"

ENV = os.environ.copy()

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("SPARK_HOME =", os.environ["SPARK_HOME"])
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])

run("java -version", env=ENV)
run(f"{SPARK_DIR}/bin/spark-submit --version", env=ENV)


RUN: apt-get -qq update
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

RUN: apt-get install -y openjdk-11-jdk-headless wget tar > /dev/null

RUN: pip -q install pyspark pandas matplotlib

RUN: wget -q -O /content/spark-3.5.1-bin-hadoop3.tgz https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz

RUN: tar -xzf /content/spark-3.5.1-bin-hadoop3.tgz -C /content/

JAVA_HOME = /usr/lib/jvm/java-11-openjdk-amd64
SPARK_HOME = /content/spark-3.5.1-bin-hadoop3
HADOOP_HOME = /content/spark-3.5.1-bin-hadoop3
RUN: java -version
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)

RUN: /content/spark-3.5.1-bin-hadoop3/bin/spark-submit --version
Welcome to
      ____              __
     / __/__  ___ _____/ /__
 

CompletedProcess(args='/content/spark-3.5.1-bin-hadoop3/bin/spark-submit --version', returncode=0, stdout="Welcome to\n      ____              __\n     / __/__  ___ _____/ /__\n    _\\ \\/ _ \\/ _ `/ __/  '_/\n   /___/ .__/\\_,_/_/ /_/\\_\\   version 3.5.1\n      /_/\n                        \nUsing Scala version 2.12.18, OpenJDK 64-Bit Server VM, 11.0.30\nBranch HEAD\nCompiled by user heartsavior on 2024-02-15T11:24:58Z\nRevision fd86f85e181fc2dc0f50a096855acf83a6cc5d9c\nUrl https://github.com/apache/spark\nType --help for more information.\n")

In [ ]:
# Step 2: Start SparkSession safely
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("PySparkNeuralNetworkColabFixed")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

sc = spark.sparkContext
print("Spark version:", spark.version)
print("Spark master:", sc.master)


TypeError: 'JavaPackage' object is not callable

## Step 3: Create sample flower dataset

This is a small Iris-style dataset with 4 numeric features and 3 classes:
- setosa
- versicolor
- virginica


In [ ]:
data = [
    (5.1, 3.5, 1.4, 0.2, "setosa"),
    (4.9, 3.0, 1.4, 0.2, "setosa"),
    (4.7, 3.2, 1.3, 0.2, "setosa"),
    (4.6, 3.1, 1.5, 0.2, "setosa"),
    (5.0, 3.6, 1.4, 0.2, "setosa"),
    (5.4, 3.9, 1.7, 0.4, "setosa"),
    (7.0, 3.2, 4.7, 1.4, "versicolor"),
    (6.4, 3.2, 4.5, 1.5, "versicolor"),
    (6.9, 3.1, 4.9, 1.5, "versicolor"),
    (5.5, 2.3, 4.0, 1.3, "versicolor"),
    (6.5, 2.8, 4.6, 1.5, "versicolor"),
    (5.7, 2.8, 4.5, 1.3, "versicolor"),
    (6.3, 3.3, 6.0, 2.5, "virginica"),
    (5.8, 2.7, 5.1, 1.9, "virginica"),
    (7.1, 3.0, 5.9, 2.1, "virginica"),
    (6.3, 2.9, 5.6, 1.8, "virginica"),
    (6.5, 3.0, 5.8, 2.2, "virginica"),
    (7.6, 3.0, 6.6, 2.1, "virginica"),
]

columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "label"]
df = spark.createDataFrame(data, columns)

print("Row count:", df.count())
df.show(truncate=False)


## Step 4: Prepare features and labels


In [ ]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml import Pipeline

assembler = VectorAssembler(
    inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],
    outputCol="features"
)

indexer = StringIndexer(inputCol="label", outputCol="labelIndex")

pipeline_prep = Pipeline(stages=[assembler, indexer])
prepared_model = pipeline_prep.fit(df)
prepared_df = prepared_model.transform(df)

prepared_df.select("label", "labelIndex", "features").show(10, truncate=False)

print("""
Explanation:
- features: the 4 flower measurements combined into one vector
- labelIndex: Spark converts flower names into numeric classes
""")


## Step 5: Split into training and test data


In [ ]:
train_df, test_df = prepared_df.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_df.count())
print("Test rows:", test_df.count())


## Step 6: Build and train the neural network

Network structure:
- 4 input neurons
- 5 hidden neurons
- 3 output neurons


In [ ]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

layers = [4, 5, 3]

mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="labelIndex",
    layers=layers,
    maxIter=150,
    blockSize=16,
    seed=42
)

mlp_model = mlp.fit(train_df)
print("Neural network model trained successfully.")

print("""
Explanation:
layers = [4, 5, 3]
- 4 input neurons because there are 4 flower features
- 5 hidden neurons in one hidden layer
- 3 output neurons because there are 3 flower classes
""")


## Step 7: Run predictions


In [ ]:
predictions = mlp_model.transform(test_df)

predictions.select(
    "label",
    "labelIndex",
    "prediction",
    "probability",
    "features"
).show(truncate=False)

print("""
Explanation:
- prediction: the class predicted by the neural network
- probability: confidence scores across all 3 classes
""")


## Step 8: Evaluate the model


In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="labelIndex",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="labelIndex",
    predictionCol="prediction",
    metricName="f1"
)

accuracy = accuracy_eval.evaluate(predictions)
f1 = f1_eval.evaluate(predictions)

print("Accuracy:", accuracy)
print("F1 score:", f1)

print("""
Explanation:
- accuracy: fraction of correct predictions
- f1: balance between precision and recall
""")


## Step 9: Show confusion-style summary


In [ ]:
predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

print("""
Explanation:
This summary shows how many rows of each true label were assigned to each predicted class.
It works like a simple confusion-matrix view.
""")


## Step 10: Convert to pandas for a simple chart


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pdf = predictions.select("label", "prediction").toPandas()
pdf


In [ ]:
summary_df = pdf.groupby(["label", "prediction"]).size().reset_index(name="count")
print(summary_df)

plt.figure(figsize=(8, 5))
plt.bar(range(len(summary_df)), summary_df["count"])
plt.xticks(
    range(len(summary_df)),
    [f"{r['label']}→{int(r['prediction'])}" for _, r in summary_df.iterrows()],
    rotation=45
)
plt.title("Prediction Count by Actual Label")
plt.xlabel("Actual Label to Predicted Class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

print("""
Explanation:
The bar chart shows how often each real flower label was mapped to each predicted class.
""")


## Step 11: Save prediction output


In [ ]:
output_pdf = predictions.select("label", "labelIndex", "prediction", "probability").toPandas()
output_path = "/content/pyspark_nn_predictions_fixed.csv"
output_pdf.to_csv(output_path, index=False)

print("Saved prediction output to:", output_path)
output_pdf.head()


## Step 12: Explanation of outputs


In [ ]:
print("""
Explanation:

1. features
   - This is the assembled numeric feature vector used by the neural network.

2. labelIndex
   - Spark converts text labels into numeric classes.

3. layers = [4, 5, 3]
   - 4 input neurons because we have 4 input features
   - 5 hidden neurons in one hidden layer
   - 3 output neurons because we have 3 classes

4. prediction
   - The predicted numeric class from the neural network.

5. probability
   - The probability distribution across the 3 classes.

6. accuracy and f1
   - These measure model performance on the test set.

7. CSV output
   - Saves the final predictions for later review or submission.
""")


## Step 13: Final conclusion


In [ ]:
print("""
This notebook demonstrates a runnable neural network in PySpark using
MultilayerPerceptronClassifier.

It includes:
- reliable Java + Spark environment setup
- Spark startup
- feature engineering
- label indexing
- train/test split
- model training
- prediction
- evaluation
- charting
- CSV export

This is the main built-in neural network option available in PySpark MLlib for classification tasks.
""")
